In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

In [2]:
def encodeLabel(data, feature):
    encoder = LabelEncoder()
    data[feature] = encoder.fit_transform(data[feature].fillna('none'))
    return data

In [3]:
profile = pd.read_csv('../data/data/customer_demographics.csv')
profile = encodeLabel(profile, 'age_range')
profile = encodeLabel(profile, 'marital_status')
profile = encodeLabel(profile, 'no_of_children')
profile = encodeLabel(profile, 'family_size')

In [4]:
tranx = pd.read_csv('../data/data/customer_transaction_data.csv')
tranx['total_discount'] = tranx['other_discount'] + tranx['coupon_discount']
tranx['other_perc'] = -1. * tranx['other_discount'] / tranx['selling_price']
tranx['coupon_perc'] = -1. * tranx['coupon_discount'] / tranx['selling_price']

In [5]:
item = pd.read_csv('../data/data/item_data.csv')
item = encodeLabel(item, 'brand_type')
item = encodeLabel(item, 'category')

In [6]:
tranx = tranx.merge(item, on='item_id')

In [7]:
feat_1 = tranx.groupby('customer_id')['item_id'].agg(['count', pd.Series.nunique])
feat_1 = feat_1.rename(columns={'count':'cust_num_tranx', 'nunique':'cust_unq_item'})
feat_1 = feat_1.reset_index()

In [8]:
feat_2 = tranx.groupby('customer_id')['selling_price'].mean().reset_index()
feat_2 = feat_2.rename(columns={'selling_price':'cust_avg_price'})

In [9]:
feat_3 = tranx.groupby('customer_id')['other_discount'].agg(['sum',np.count_nonzero]).reset_index()
feat_3 = feat_3.rename(columns={'sum':'cust_odsc_sum', 'count_nonzero':'cust_odsc_cnt'})

In [10]:
feat_4 = tranx.groupby('customer_id')['brand'].agg([pd.Series.nunique]).reset_index()
feat_4 = feat_4.rename(columns={'nunique':'cust_unq_brand'})

In [11]:
feat_5 = tranx.groupby('customer_id')['brand_type'].agg([pd.Series.nunique]).reset_index()
feat_5 = feat_5.rename(columns={'nunique':'cust_unq_brand_typ'})

In [12]:
feat_6 = tranx.groupby('customer_id')['category'].agg([pd.Series.nunique]).reset_index()
feat_6 = feat_6.rename(columns={'nunique':'cust_unq_category'})

In [13]:
feat_7 = tranx.groupby('customer_id')['coupon_discount'].agg(['sum',np.count_nonzero]).reset_index()
feat_7 = feat_7.rename(columns={'sum':'cust_cdsc_sum', 'count_nonzero':'cust_cdsc_cnt'})

In [14]:
feat_8 = tranx.groupby('customer_id')['selling_price'].sum().reset_index()
feat_8 = feat_8.rename(columns={'selling_price':'cust_sum_price'})

In [15]:
profile = profile.merge(feat_1, on='customer_id', how='outer')
profile = profile.merge(feat_2, on='customer_id', how='outer')
profile = profile.merge(feat_3, on='customer_id', how='outer')
profile = profile.merge(feat_4, on='customer_id', how='outer')
profile = profile.merge(feat_5, on='customer_id', how='outer')
profile = profile.merge(feat_6, on='customer_id', how='outer')
profile = profile.merge(feat_7, on='customer_id', how='outer')
profile = profile.merge(feat_8, on='customer_id', how='outer')
profile = profile.fillna(-1)

In [16]:
driver = pd.read_csv('../data/driver.csv')[['id','customer_id']]
data = driver.merge(profile, on='customer_id', how='left').drop('customer_id',axis=1)
data = data.fillna(-1)

In [17]:
data.to_csv('../data/feature/customer_feature.csv', index=False)

In [18]:
data.shape

(128595, 18)

In [19]:
data.head()

,id,age_range,marital_status,rented,family_size,no_of_children,income_bracket,cust_num_tranx,cust_unq_item,cust_avg_price,cust_odsc_sum,cust_odsc_cnt,cust_unq_brand,cust_unq_brand_typ,cust_unq_category,cust_cdsc_sum,cust_cdsc_cnt,cust_sum_price
0,1,3.0,2.0,0.0,0.0,3.0,5.0,310,208,184.260484,-10282.37,167.0,84,2,8,-89.05,1.0,57120.75
1,2,2.0,0.0,0.0,1.0,3.0,3.0,385,244,234.247013,-10664.18,209.0,102,2,12,-1237.79,12.0,90185.10
2,6,3.0,0.0,0.0,1.0,3.0,7.0,970,533,121.094495,-17261.79,457.0,186,2,11,-2145.72,85.0,117461.66
3,7,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,237,216,98.276034,-3947.37,120.0,73,2,8,-178.10,2.0,23291.42
4,9,3.0,0.0,0.0,1.0,3.0,3.0,562,327,120.636103,-11534.90,281.0,106,2,8,-265.01,10.0,67797.49
